# Phase5: 入出庫シミュレーション — AIを「システム」の一部品にする

> ⏱ 所要時間: 説明を読む 30分 ＋ 実行 約1分

## このプロジェクトのゴール（おさらい）
駐車場のナンバー読み取り精算システムのミニ版を作り、**AIはシステムの一部品**であることを確かめます。ここまでの積み上げです。

| Phase | 作ったもの（AIの部分） |
|---|---|
| 1〜3 | 板を**見つけ**（検出）、文字を**読む**（OCR）。AIの「目」 |
| 4 | 読み取りを**検査**し、**自信**を測り、「自動処理／要確認／読み取り不可」に**振り分ける** |

## このNotebookのゴール
駐車場システムの**残りの部分**（従来のソフトウェア）を作り、**システム全体**で測ります。

1. **入庫**を記録し（データベース）、**出庫**で記録と**照合**し、**料金を計算**する
2. AIの読み取りに**間違いが混ざったまま**、システムが動くとどうなるか
3. 測るのは、AIの精度ではなく、**システムの結果**（誤課金・人手の負担・自動精算できた割合）

> 💡 **設計の原則（再掲）**: 誤って課金するより、**「要確認」に回す**ほうを優先する。ただし、要確認は**人手**がかかる。**そのバランス**を、数字で見ます。

## この回で分かること（先取り）
- AIの「自信満々の誤り」は、**データベースとの照合**で**ほとんど止まる**（AIの外側が、AIを助ける）
- それでも**誤課金は完全にはゼロにならない**（似たナンバーの車が同時にいると）
- **誤課金をゼロに近づける**ほど**人手が大きく増える**。どこで折り合うかは、**損害の重さ（コスト）**で決まる

## キーワード
| 用語 | 意味 |
|---|---|
| **入庫／出庫** | 車が駐車場に入る／出る |
| **照合** | 出庫のとき、読み取ったナンバーを、入庫の記録と突き合わせること |
| **一意に一致** | 入庫中の記録の中に、ちょうど1件だけ一致すること |
| **近傍（きんぼう）** | 1文字違いなど、よく似た別のナンバー（の記録） |
| **誤課金** | 別の車の記録で課金してしまうこと。**最も避けたい失敗** |
| **自動精算率** | 人手なしで、正しく精算できた割合 |
| **人手対応率** | 人が確認・入力する必要があった割合（要確認＋読み取り不可・未照合） |
| **SQLite** | ファイル1つで使える、軽いデータベース（SQL言語で操作） |
| **期待コスト** | 1台あたりの平均的な損失。人手対応×1＋誤課金×N（Nは誤課金の重さ） |

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch, PIL, numpy, pandas  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"必要なライブラリが見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (plate_reader_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

from parking_sim.db import ParkingDB
from parking_sim.fees import fee_yen
from parking_sim.policy import POLICIES, ExitPolicy, process_exit
from parking_sim.setup import auto_threshold, to_events
from parking_sim.simulate import simulate
from parking_sim.world import make_world
from rules.decision import AUTO, REJECT, REVIEW

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
%matplotlib inline
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").exists())
SIM = ROOT / "outputs" / "sim"
if not (SIM / "events_S2_s0.csv").exists():
    print("シミュレーションの結果がありません。ターミナルで  .venv/bin/python src/run_phase5.py  を実行してください（約15分）。以降の「本実験」の節は、結果ができてから動きます。")
def load(sc, s):
    return pd.read_csv(SIM / f"events_{sc}_s{s}.csv", keep_default_na=False)

上のセルは、道具の読み込みです。エラーが出たら、右上のカーネルを **Python (plate_reader_lab)** に切り替えてください。

## 1. システムの全体像

```
【入庫】 カメラ → AIが読む → ┬ 自動処理 → そのまま記録（読み間違いも、そのまま記録される！）
                           └ 要確認／読み取り不可 → 人が正しいナンバーを入力 → 記録

【出庫】 カメラ → AIが読む → データベースと照合 → ┬ 1件だけ一致 かつ 自信あり → 自動精算
                                                  ├ 近いナンバーの記録あり → 要確認（候補を添えて人へ）
                                                  └ 記録が見つからない・読めない → 人が手で探す（読み取り不可・未照合）
```

**ここが大事**: 入庫で読み間違いが**そのまま記録される**と、出庫で**正しく**読んでも一致しません。AIの誤りは、**あとで別の形で現れる**のです。

## 2. 料金表

簡単な料金表を使います: **最初の30分は無料、以降は30分ごとに100円、1日（24時間）の上限は1,500円**。

In [ ]:
rows = [{"駐車時間(分)": m, "料金(円)": fee_yen(m)} for m in [10, 30, 31, 60, 90, 180, 600, 1440, 1500, 3000]]
display(pd.DataFrame(rows).set_index("駐車時間(分)").T)
xs = np.arange(0, 3001, 5)
plt.figure(figsize=(7, 2.8)); plt.step(xs, [fee_yen(x) for x in xs], where="post"); plt.grid(alpha=0.3)
plt.xlabel("駐車時間（分）"); plt.ylabel("料金（円）"); plt.title("料金表（30分無料・30分100円・1日上限1,500円）");

**見るポイント**: 最初の30分は0円、その後は階段状に増え、**1,500円で頭打ち**（600分あたり）。24時間ごとに、上限がまた積み重なります。

**なぜ料金が大事か**: 誤課金のとき、**課金額の誤差**は、**2台の駐車時間の差**で決まります。短時間の車と長時間の車を取り違えると、誤差が大きくなります。

## 3. データベース（SQLite）— 入庫の記録

入庫の記録は、**表（テーブル）**に1行ずつ保存します。ここでは、メモリ上の小さなデータベースを作ります。

In [ ]:
db = ParkingDB()
print(db.conn.execute("SELECT sql FROM sqlite_master WHERE type='table'").fetchone()[0])
# 3台が入庫した（plate=読み取ったナンバー、conf=確信度、truth_vehicle=評価用の正解の車番号）
A, B, C = "沖縄300あ12-34", "沖縄300あ12-84", "品川500か77-21"      # AとBは、4桁目の「3」と「8」だけ違う
db.enter(A, 0.95, AUTO, 0.0, 0); db.enter(B, 0.93, AUTO, 5.0, 1); db.enter(C, 0.97, AUTO, 8.0, 2)
pd.read_sql("SELECT id, plate, conf, entered_at, status, truth_vehicle FROM entries", db.conn)

**表の見方**: `plate` が読み取ったナンバー、`status='in'` は駐車中、`truth_vehicle` は**評価のためだけ**に持つ「本当はどの車か」（照合の処理では使いません）。

出庫のときの**検索**は、次の2種類です。
- **完全一致**: `WHERE plate = '読んだナンバー'`
- **近傍**: `edit(plate, '読んだナンバー') <= 1`（**1文字以内の違い**。「編集距離」という指標）

In [ ]:
print("完全一致で A を探す :", [r[0] for r in db.active_exact(A)], "（記録id）")
print("Aに近い記録（1文字違い以内）:", [(r[0], r[3]) for r in db.active_near(A, 1)])
print("Cに近い記録 :", [(r[0], r[3]) for r in db.active_near(C, 1)])

**見るポイント**: Aに近い記録として、**A自身とB**が出てきます（1文字違い）。**AとBは、形の似た数字（3と8）の違いだけ**なので、AIが取り違えやすい組み合わせです。Cは、似た記録がありません。

## 4. 出庫の照合方針 — 4つの設計

出庫のとき、**自動精算してよいか**を決めるルールが、設計の中心です。4通りを比べます。

| 方針 | ルール | ねらい |
|---|---|---|
| **P1 保守** | AIが「自動処理」と判断し、かつ、記録に**一意に一致**したときだけ自動精算 | 基本形 |
| **P2 照合を信頼** | AIの確信度が低くても、**書式が正しく**、記録に一意に一致すれば自動精算 | 人手を減らす（データベースとの一致は強い証拠と考える） |
| **P3 保守＋近傍チェック** | P1に加え、一致した記録の**近くに別の記録（1文字違い）**があれば、**要確認** | 取り違えを防ぐ（似た車が同時にいる場合） |
| **P4 照合を信頼＋近傍チェック** | P2＋近傍チェック | 両方 |

それ以外（一致する記録なし、近い記録あり、読み取り不可）は、**人が確認**します。人は、**車や画像を見て**正しい記録を選べるものとします（人の誤りは考えない）。

## 5. 小さな実例で動きを見る

先ほどのデータベース（A, B, C）で、**同じ出庫**を4つの方針で処理します。

- **ケース1**: Aの車が出庫。AIは**正しく**読み、自信がある
- **ケース2**: Aの車が出庫。AIは**Bのナンバーと読み間違え**、しかも**自信満々**（形の似た3と8の取り違え）
- **ケース3**: Aの車が出庫。AIは正しく読んだが、**自信が低い**

In [ ]:
def run_case(text, decision, conf, truth=0, t_out=120.0):
    out = {}
    for p in POLICIES:
        d = ParkingDB()
        d.enter(A, 0.95, AUTO, 0.0, 0); d.enter(B, 0.93, AUTO, 5.0, 1); d.enter(C, 0.97, AUTO, 8.0, 2)
        o = process_exit(d, text, True, decision, conf, truth, t_out, p)
        out[p.name[:2]] = f"{o.action}" + ("" if o.action != "自動精算" else (" ✔正しい" if o.correct else " ✘誤課金！"))
    return out

cases = {"ケース1: 正しく読めた・自信あり": run_case(A, AUTO, 0.99),
         "ケース2: Bと読み間違え・自信満々": run_case(B, AUTO, 0.99),
         "ケース3: 正しく読めた・自信が低い": run_case(A, REVIEW, 0.30)}
pd.DataFrame(cases).T

**見るポイント**
- **ケース1**: **P1・P2は自動精算。P3・P4は、正しく読めているのに「要確認」**にします。Aの**近くにBの記録がある**ためです。**近傍チェックは、似た車がいる限り、正しい読み取りまで人に回す**。これが、あとで見る「人手が大きく増える」理由です。
- **ケース2（重要）**: **P1とP2は、Bの記録に課金してしまいます（✘誤課金！）**。AIは自信満々なので、止まりません。**P3とP4は、「Bの近くにAの記録がある」ので、要確認に回し**、誤課金を防ぎます。
- **ケース3**: P1は**人が確認**（保守的）、P2は**自動精算**（記録に一意に一致するので、信頼）、P3・P4は、近傍チェックで**要確認**。

### 誤課金の「二次被害」
誤課金が起きると、**もう1つ問題が起きます**。Bの記録が**誤って閉じられ**、Aの記録が**「入庫中」のまま残る**のです。あとで**Bの車が出庫**すると、どうなるでしょう。

In [ ]:
d = ParkingDB()
d.enter(A, 0.95, AUTO, 0.0, 0); d.enter(B, 0.93, AUTO, 5.0, 1)
o1 = process_exit(d, B, True, AUTO, 0.99, 0, 120.0, POLICIES[0])      # Aの出庫を、Bと読み間違えて精算（誤課金）
print("1回目（Aの出庫）:", o1.action, "／ 正しい車か:", o1.correct, "／ 課金した車の番号:", o1.billed_vehicle)
o2 = process_exit(d, B, True, AUTO, 0.99, 1, 200.0, POLICIES[0])       # その後、Bの車が出庫
print("2回目（Bの出庫）:", o2.action, "／", o2.note)
print("入庫中のまま残っている記録の数:", d.n_active(), "（Aの記録が閉じられずに残った）")

**見るポイント**: Bが出庫するとき、**Bの記録は既に閉じられて**います。人が**復元**して精算できましたが、**手間がかかり**、データベースには**Aの記録が残ったまま**（実際の駐車場では、「ずっと駐車中の車」が増える）。**誤課金は、1件で終わらない**のです。

## 6. シミュレーションの世界

車を何千台も入出庫させて、システムの性能を測ります。世界の設定は次のとおりです。

- 車は**1分に0.5台**の割合で到着し、駐車時間は**短い（70%・中央値1時間）・中くらい（25%・4時間）・長い（5%・1日以上）**
- **似たナンバーの車**（車列、レンタカーの連番など）を、割合 `similar_fraction` で混ぜる。**形の似た数字の1文字違い**（6↔8、5↔6 など、AIが取り違えやすいもの）
- 入庫と出庫の**撮影条件**（Easy/Medium/Hard）は、車ごと・入出庫ごとに独立に抽選

次のセルで、世界の様子を見ます（似たナンバー30%）。

In [ ]:
w = make_world(3000, 0, similar_fraction=0.3)
stay = np.array([v.t_out - v.t_in for v in w])
ts = np.linspace(0, max(v.t_out for v in w[:2800]), 400)
active = [sum(v.t_in <= t < v.t_out for v in w) for t in ts]
pairs = sum(1 for i in range(1, len(w)) for j in range(max(0, i - 6), i)
            if len(w[i].label["text"]) == len(w[j].label["text"]) and sum(a != b for a, b in zip(w[i].label["text"], w[j].label["text"])) == 1)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
ax[0].hist(np.clip(stay, 0, 600), bins=40); ax[0].set(title="駐車時間の分布（600分で打ち切り）", xlabel="分")
ax[1].plot(ts / 60, active); ax[1].set(title="同時に駐車している台数", xlabel="経過時間（時間）")
ax[2].hist([fee_yen(s) for s in stay], bins=30); ax[2].set(title="料金の分布", xlabel="円")
fig.tight_layout()
print(f"車 {len(w)}台 ／ 同時駐車は最大約 {max(active)} 台 ／ 近い入庫順の車との間で、1文字違いのナンバーの組: {pairs} 組")
print("ナンバーの例:", [v.label["text"] for v in w[:3]], " 似たナンバーの例:", [(w[i].label["text"], w[j].label["text"]) for i in range(1, 60) for j in range(max(0, i - 6), i) if len(w[i].label["text"]) == len(w[j].label["text"]) and sum(a != b for a, b in zip(w[i].label["text"], w[j].label["text"])) == 1][:2])

**見るポイント**: 同時駐車は**最大130台ほど**、似たナンバーの組が**たくさん**（上の例では約900組）あります。**AIが取り違えやすい数字の違い**で作ってあるので、**誤課金が起きやすい、厳しい設定**です。

## 7. 本実験の結果

次の4つの場面を、**同じ車・同じAIの読み取り結果**に対して、4つの方針で比べます（**対等な比較**）。各場面、**4,000台×3回＝12,000台**の出庫です。

| 場面 | 内容 |
|---|---|
| **S1** | ナンバーはランダム（似た車は偶然だけ）、昼間中心（Easy 60%・Medium 30%・Hard 10%） |
| **S2** | 似たナンバーが**30%**、昼間中心 |
| **S3** | 似たナンバー30%、**悪条件が多い**（Easy・Medium・Hardが同数） |
| **S4** | 似たナンバーが**80%**（車列など）、昼間中心【**ストレステスト**】 |

自動処理のしきい値は、Phase4の方法（較正セット・昼間中心・目標の誤り率1%）で決めた値 τ=0.509 を、**全場面で共通**に使います（運用では、場面ごとにしきい値を変えられないため。S3は、**設計の想定とずれた**場面です）。

**表の見方**: 割合は、**出庫した車の台数**に対する割合です。誤課金の件数も載せます（**件数が少ないので、傾向として読んでください**）。

### S1: ナンバーはランダム

| 方針 | 自動精算率 | **誤課金率** | 誤課金の件数 | 要確認率 | 読み取り不可・未照合率 | 人手対応率 | 期待コスト |
|---|---|---|---|---|---|---|---|
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.8443 | **0.0000** ±0.0000 | 0／12000件 | 0.0628 | 0.0929 | 0.1557 | 0.1557 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.8755 | **0.0000** ±0.0000 | 0／12000件 | 0.0316 | 0.0929 | 0.1245 | 0.1245 |
| P3 保守＋近傍チェック | 0.8443 | **0.0000** ±0.0000 | 0／12000件 | 0.0628 | 0.0929 | 0.1557 | 0.1557 |
| P4 照合を信頼＋近傍チェック | 0.8755 | **0.0000** ±0.0000 | 0／12000件 | 0.0316 | 0.0929 | 0.1245 | 0.1245 |

### S2: 似たナンバー30%

| 方針 | 自動精算率 | **誤課金率** | 誤課金の件数 | 要確認率 | 読み取り不可・未照合率 | 人手対応率 | 期待コスト |
|---|---|---|---|---|---|---|---|
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.8404 | **0.0001** ±0.0001 | 1／12000件 | 0.0580 | 0.1015 | 0.1595 | 0.1612 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.8693 | **0.0002** ±0.0001 | 2／12000件 | 0.0290 | 0.1016 | 0.1306 | 0.1339 |
| P3 保守＋近傍チェック | 0.6225 | **0.0000** ±0.0000 | 0／12000件 | 0.2761 | 0.1014 | 0.3775 | 0.3775 |
| P4 照合を信頼＋近傍チェック | 0.6432 | **0.0000** ±0.0000 | 0／12000件 | 0.2553 | 0.1014 | 0.3568 | 0.3568 |

### S3: 似たナンバー30%・悪条件が多い

| 方針 | 自動精算率 | **誤課金率** | 誤課金の件数 | 要確認率 | 読み取り不可・未照合率 | 人手対応率 | 期待コスト |
|---|---|---|---|---|---|---|---|
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.6195 | **0.0000** ±0.0000 | 0／12000件 | 0.0988 | 0.2817 | 0.3805 | 0.3805 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.6608 | **0.0003** ±0.0001 | 4／12000件 | 0.0569 | 0.2819 | 0.3388 | 0.3455 |
| P3 保守＋近傍チェック | 0.4593 | **0.0000** ±0.0000 | 0／12000件 | 0.2590 | 0.2817 | 0.5407 | 0.5407 |
| P4 照合を信頼＋近傍チェック | 0.4893 | **0.0000** ±0.0000 | 0／12000件 | 0.2290 | 0.2817 | 0.5107 | 0.5107 |

### S4: 似たナンバー80%（ストレス）

| 方針 | 自動精算率 | **誤課金率** | 誤課金の件数 | 要確認率 | 読み取り不可・未照合率 | 人手対応率 | 期待コスト |
|---|---|---|---|---|---|---|---|
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.8302 | **0.0002** ±0.0001 | 2／12000件 | 0.0665 | 0.1031 | 0.1696 | 0.1729 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.8657 | **0.0007** ±0.0004 | 8／12000件 | 0.0301 | 0.1036 | 0.1337 | 0.1470 |
| P3 保守＋近傍チェック | 0.3725 | **0.0000** ±0.0000 | 0／12000件 | 0.5246 | 0.1029 | 0.6275 | 0.6275 |
| P4 照合を信頼＋近傍チェック | 0.3885 | **0.0000** ±0.0000 | 0／12000件 | 0.5086 | 0.1029 | 0.6115 | 0.6115 |

### 何が分かるか

1. **誤課金は、とても少ない。** S1（ランダム）は **0件／12,000件**。S2も、P1で**1件**。AIの読み取りは完璧ではない（Phase4で、自動処理の誤りは約1%）のに、**システムとしての誤課金は、その100分の1以下**です。
   **読み取りの「自信満々の誤り」と、誤課金の件数**:

- S2 似たナンバー30%・昼間中心: 出庫の読み取りで「自動処理」と判断された 10208 件のうち、読み間違いは 57 件（0.56%）。一方、システム全体の誤課金は 1 件（出庫 12000 件に対して 0.0083%）。
- S4 似たナンバー80%（車列など）・昼間中心【ストレス】: 出庫の読み取りで「自動処理」と判断された 10074 件のうち、読み間違いは 62 件（0.62%）。一方、システム全体の誤課金は 2 件（出庫 12000 件に対して 0.0167%）。

   → **データベースとの照合が、AIの自信満々の誤りの、ほとんどを止めています。** AIが読み間違えたナンバーは、**入庫記録に存在しない**ことが多く、**一致しない → 人が確認**に回るからです（Phase4の「残る誤り」の予告どおり）。**AIの外側（データベース）が、AIを助けている**。
2. **誤課金が起きるのは、「似たナンバーの車が同時にいて、AIが**そのナンバーそっくりに**読み間違えた」とき。** 似たナンバーが多いほど（S2 → S4）、増える。**悪条件が多い場面（S3）では、P2で4件**と、最も多い。
3. **人手対応率は、約13〜16%（S1・S2）、悪条件が多いと約34〜38%（S3）。** AIが読めない画像（Hard、読み取り不可）が、大きな割合を占める。**カメラ・照明の改善**が、人手を減らす最大の手段になりそうです（あとで内訳を見ます）。
4. **P2（照合を信頼）は、P1より人手が少ない**（0.16→0.13）。**代わりに、誤課金が少し増える**（S4: 2件→8件）。
5. **近傍チェック（P3・P4）は、誤課金を0にする。** **しかし、人手が2〜4倍に増える**（S2で0.16→0.38、S4で0.17→0.63）。「近くに似た記録がある」場面が多いので、**要確認が大量に出る**ためです。

**期待コスト**（1台あたり。人手対応=1、誤課金=20）は、S2では **P2（0.134）< P1（0.161）< P4（0.357）< P3（0.378）**。**誤課金の重さが20倍程度なら、近傍チェックは割に合いません。**

## 8. 人手に回った理由を調べる

「人手対応」の中身は何でしょう。S2のP1の結果（3回合計、12,000台の出庫）を、**理由別・撮影条件別**に分けます。

In [ ]:
from collections import Counter
tau = auto_threshold(0.01)
dfs = []
for s in range(3):
    r, df = simulate(to_events(load("S2", s), tau), POLICIES[0], 20.0, detail=True); dfs.append(df)
df = pd.concat(dfs, ignore_index=True)
reason = df[df.action != "自動精算"].note.str.replace("（先の誤課金で記録が閉じられていた）", "", regex=False).value_counts()
print(f"出庫 {len(df)}台のうち、人手に回ったのは {int((df.action != '自動精算').sum())}台（{(df.action != '自動精算').mean():.1%}）。理由の内訳:")
display((reason / len(df)).rename("全出庫に対する割合").round(4).to_frame())
cross = pd.crosstab(df.difficulty, df.action).loc[["easy", "medium", "hard"]]
display((cross.div(cross.sum(axis=1), axis=0)).round(3))

**見るポイント**
- **「候補なし（未照合）」「読み取り不可」が大きな割合**（合計で人手の約3分の2）。これは、**AIが読めなかった**（書式が合わない、または大きく間違えた）ケースです。
- **「確信度が低いため確認」「近い候補あり」**が、残りの約3分の1。
- **撮影条件別**（下の表）: **Hard の約98%が人手**に回り、**Medium は約14%、Easy は約3%**。**人手の大部分は、Hard（暗い・ぼける・汚れる）の画像**（人手に回った1,914台のうち、Hardだけで約6割）。
- → **AIをこれ以上賢くするより、撮影の条件（カメラ・照明・設置）を改善する**ほうが、人手を減らす効果が大きい、と分かります。**システム全体で見ないと見えない**ことです。

## 9. 誤課金の重さで、最適な方針は変わる

どの方針が良いかは、**誤課金1件の損害（N）が、要確認1件（=1）の何倍か**で決まります。S4（ストレス）の結果で、Nを動かして、**期待コスト**を比べます。

In [ ]:
res = {p.name[:2]: [] for p in POLICIES}
for s in range(3):
    ev = to_events(load("S4", s), tau)
    for p in POLICIES:
        res[p.name[:2]].append(simulate(ev, p, 20.0))
m = {k: (np.mean([r["人手対応率"] for r in v]), np.mean([r["誤課金率"] for r in v])) for k, v in res.items()}
Ns = np.array([1, 3, 10, 30, 100, 300, 1000, 3000, 10000])
plt.figure(figsize=(7.5, 3.8))
for k, (h, b) in m.items():
    plt.plot(Ns, h + b * Ns, "o-", label=f"{k}（人手{h:.2f}・誤課金{b * 100:.3f}%）")
plt.xscale("log"); plt.yscale("log"); plt.grid(alpha=0.3, which="both"); plt.legend(fontsize=8)
plt.xlabel("誤課金1件の重さ N（要確認1件=1）"); plt.ylabel("1台あたりの期待コスト"); plt.title("S4（ストレス）: 誤課金の重さと期待コスト")
h1, b1 = m["P1"]; h2, b2 = m["P2"]; h3, b3 = m["P3"]
print(f"P2がP1より得なのは、N が約 {(h1 - h2) / (b2 - b1):.0f} 以下のとき ／ P3がP1より得になるのは、N が約 {(h3 - h1) / (b1 - b3):.0f} 以上のとき")

**見るポイント**
- **Nが小さい**（誤課金がそれほど痛くない）なら、**人手が少ない P2** が最良。
- **Nが約70を超える**と、**P1（保守）**のほうが得に。**Nが約2,700を超える**と、**近傍チェック（P3）**が得になる。
- つまり、「近傍チェックをするか」は、**技術の問題ではなく、誤課金の損害をどう見積もるか**で決まります。**お客さまとの信頼、返金対応、クレーム対応のコスト**を含めて、**運用する側が決める**。
- **ほとんどの運用では、P1〜P2の間**に答えがある（誤課金が約2,700倍も重いとは、普通は言えない）。

### 目標の誤り率を変えると（S4、3回平均）
自動処理のしきい値（Phase4の「目標の誤り率」）を変えると、人手と誤課金はどう動くでしょう。

| 方針 | 目標の誤り率 | しきい値 | 誤課金率 | 人手対応率 | 期待コスト |
|---|---|---|---|---|---|
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.002 | 0.707 | 0.0000 | 0.3068 | 0.3068 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.002 | 0.707 | 0.0007 | 0.1305 | 0.1438 |
| P3 保守＋近傍チェック | 0.002 | 0.707 | 0.0000 | 0.6887 | 0.6887 |
| P4 照合を信頼＋近傍チェック | 0.002 | 0.707 | 0.0000 | 0.6108 | 0.6108 |
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.010 | 0.509 | 0.0002 | 0.1696 | 0.1729 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.010 | 0.509 | 0.0007 | 0.1337 | 0.1470 |
| P3 保守＋近傍チェック | 0.010 | 0.509 | 0.0000 | 0.6275 | 0.6275 |
| P4 照合を信頼＋近傍チェック | 0.010 | 0.509 | 0.0000 | 0.6115 | 0.6115 |
| P1 保守（AIが自動処理と判断＋一意に一致） | 0.050 | 0.010 | 0.0008 | 0.1693 | 0.1843 |
| P2 照合を信頼（書式OKで一意に一致なら自動） | 0.050 | 0.010 | 0.0008 | 0.1693 | 0.1843 |
| P3 保守＋近傍チェック | 0.050 | 0.010 | 0.0000 | 0.6192 | 0.6192 |
| P4 照合を信頼＋近傍チェック | 0.050 | 0.010 | 0.0000 | 0.6192 | 0.6192 |

In [ ]:
f = SIM / "tradeoff.png"
if f.exists():
    display(IPImage(filename=str(f)))

**見るポイント**
- **しきい値を厳しくする（目標の誤り率を小さく）**と、**人手が増え**、誤課金が減る（左の図の、左上から右下へ）。
- **近傍チェックあり（P3・P4）は、誤課金は0だが、人手が大きい**（グラフの右端）。**なし（P1・P2）は、人手は少なく、誤課金が少しある**。
- **P2は、しきい値を変えても人手があまり変わらない**（照合を信頼するため、確信度に頼らないから）。

## 🧪 試してみよう

### 演習1: 自動処理のしきい値を変える
**やること**: 下のセルの **`target = 0.01`** を **`0.002`**、**`0.05`** に書き換えて実行します（S2、3回平均）。
- **見るポイント**: 目標を**小さく（厳しく）**すると、**人手対応率**と**誤課金率**は、方針ごとにどう変わりますか？
- **分かること**: AIの**確信度のしきい値**が、システム全体の人手と誤課金に**どう効くか**。

In [ ]:
target = 0.01       # ← 0.002 / 0.05 に書き換えて実行
t_ = auto_threshold(target)
rows = []
for p in POLICIES:
    rs = [simulate(to_events(load("S2", s), t_), p, 20.0) for s in range(3)]
    rows.append({"方針": p.name[:2], "しきい値": round(t_, 3), "人手対応率": round(np.mean([r["人手対応率"] for r in rs]), 4),
                 "誤課金率": round(np.mean([r["誤課金率"] for r in rs]), 5), "期待コスト(N=20)": round(np.mean([r["期待コスト"] for r in rs]), 4)})
pd.DataFrame(rows).set_index("方針")

### 演習2: 自分で方針を設計する
**やること**: 下のセルの `ExitPolicy(...)` の設定を変えて、**S4（ストレス）**で試します。
- `auto_if_valid`: `True` にすると、確信度が低くても、書式が正しく一意に一致すれば自動精算（P2の考え方）
- `neighbor_check`: `True` にすると、近くに別の記録があれば要確認（P3の考え方）
- `near_k`: 要確認の候補にする「近さ」（編集距離）。`1` が標準、`2` にすると、より遠い候補も探す
**課題**: **人手対応率を P1 より小さくしつつ、誤課金を P2 より小さく**できる設定はありますか？
- **見るポイント**: `near_k=2` にすると、「候補なし（未照合）」が**減って「要確認」が増える**？ 人手の合計は変わる？
- **分かること**: 方針の設計は、**複数の指標（人手・誤課金）のバランス**。**一方を良くすれば、他方が悪くなる**ことが多い。

In [ ]:
mine = ExitPolicy("自作", auto_if_valid=True, neighbor_check=False, near_k=1)      # ← 設定を変えてみる
rows = []
for p in [POLICIES[0], POLICIES[1], mine]:
    rs = [simulate(to_events(load("S4", s), tau), p, 20.0) for s in range(3)]
    rows.append({"方針": p.name[:12], "自動精算率": round(np.mean([r["自動精算率"] for r in rs]), 4), "誤課金率": round(np.mean([r["誤課金率"] for r in rs]), 5),
                 "要確認率": round(np.mean([r["要確認率"] for r in rs]), 4), "読み取り不可・未照合率": round(np.mean([r["読み取り不可_未照合率"] for r in rs]), 4),
                 "期待コスト(N=20)": round(np.mean([r["期待コスト"] for r in rs]), 4)})
pd.DataFrame(rows).set_index("方針")

### 考えてみよう（コード不要）
1. **AIの自動処理の誤りは約1%**なのに、**システムの誤課金は100分の1以下**でした。**この差**は、**何が生んでいる**のでしょう？ もし**データベースが無ければ**（読み取り結果をそのまま使うなら）、誤りはどうなりますか？
2. 人手の大部分は**Hard（撮影条件が悪い）**でした。**AIを改良する**のと、**カメラ・照明を改善する**のと、**どちらにお金をかける**べきでしょうか？ どう比べますか？
3. 実際の駐車場では、**出庫ゲートで人を待たせる**ことになります。**要確認の待ち時間**は、**お客さまの不満**につながります。このシミュレーションに**足りない要素**は何でしょう？（ヒント: ゲートの渋滞、確認の待ち時間、人の誤り、夜間の無人運用）
4. **入庫でAIが自動処理して読み間違えた**ナンバー（全体の約0.6%）は、**あとで別の形で現れ**ます。それは、どんな形でしょう？ **入庫でも**確認を厳しくするべきでしょうか？

---

## ✅ まとめ

- **システム全体**（AI＋検査＋確信度＋データベース＋料金＋人の確認）を作り、**システムとしての結果**を測った。
- **データベースとの照合が、AIの自信満々の誤りの、ほとんどを止める。** AIの読み誤り（約0.6%）に対し、誤課金は**数千件に1件以下**。**AIの外側の仕組みが、AIの弱点を補う**。
- **誤課金は、「似たナンバーの車が同時にいる」ときに起きる**。**1件で終わらず、二次被害**（別の車の記録を誤って閉じる）を生む。
- **誤課金をゼロに近づけるほど、人手が増える**（近傍チェックで、人手が2〜4倍）。**どこで折り合うかは、誤課金の損害の重さ（コスト）で決まる**。運用者が決める。
- **人手の大部分は、撮影条件が悪い（Hard）画像**。**AIの改良より、カメラ・照明の改善が効く**可能性が大きい。システム全体で測らないと見えない。
- 注意: 合成画像と、単純化した世界（ゲートの渋滞・人の誤り・待ち時間は考えない）。**実運用の数字ではなく、設計の考え方**を学ぶ。

➡ 次は **Phase6: 評価・検証レポート**。これまでの全部を、**1つのレポート**にまとめます。難易度別・条件別の評価、**混同行列**（どの文字を取り違えるか）、**失敗例集**、システムとしての限界。最後に、**test データを1回だけ**使って、最終評価をします。